# Corpus Statistics (Kaggle)

Computes per-document and total corpus statistics using the **exact same
data loading and QC pipeline** as the training notebooks.

**Required input:** `alexkorablev/qom-corpus` (add via Notebook → Add Input)


In [1]:
# Cell 1: Dependencies
!pip install openpyxl sacrebleu --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.5 MB/s eta 0:00:00


In [2]:
# Cell 2: Imports and helpers
import re
import numpy as np
import pandas as pd
from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/alexkorablev/qom-corpus")
OUT_DIR  = Path("/kaggle/working")

_tok = re.compile(r"[^\s\W]+|[^\w\s]", re.UNICODE)

def tokenize(text):
    """Orthographic tokenization: whitespace + punctuation split."""
    return _tok.findall(str(text))

def seg_stats(series):
    lengths = series.dropna().apply(lambda x: len(tokenize(x)))
    return {
        "tokens":   int(lengths.sum()),
        "avg_len":  round(float(lengths.mean()), 2),
        "std_len":  round(float(lengths.std()),  2),
    }

print("Helpers OK")
print(f"DATA_DIR exists: {DATA_DIR.exists()}")


Helpers OK
DATA_DIR exists: True


In [3]:
# Cell 3: Load xlsx corpora + metadata name pairs
# Mirrors read_workbook() + read_metadata_names() from training notebook exactly.

def read_workbook(path):
    doc_name = path.stem
    xl = pd.ExcelFile(path)
    frames = []
    for sheet in xl.sheet_names:
        try:
            raw = xl.parse(sheet)
        except Exception as e:
            print(f"  [SKIP] {sheet}: {e}")
            continue
        raw.columns = [str(c).strip() for c in raw.columns]
        if "linea_qom" not in raw.columns or "linea_es" not in raw.columns:
            continue
        sub = raw[["linea_qom", "linea_es"]].copy()
        sub.columns = ["qom", "es"]
        sub = sub.dropna(subset=["qom", "es"])
        sub["qom"] = sub["qom"].astype(str).str.strip()
        sub["es"]  = sub["es"].astype(str).str.strip()
        sub = sub[(sub["qom"] != "") & (sub["es"] != "")]
        sub = sub[(sub["qom"] != "nan") & (sub["es"] != "nan")]
        sub["source_doc"] = doc_name
        frames.append(sub[["qom", "es", "source_doc"]])
    if not frames:
        print(f"  [WARN] No usable sheets in {path.name}")
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)

def read_metadata_names(path):
    """Extract nombre_*_qom/es pairs from Capítulos/Secciones/Fragmentos tabs."""
    doc_name = path.stem
    xl = pd.ExcelFile(path)
    pairs = []
    for sheet in xl.sheet_names:
        try:
            raw = xl.parse(sheet)
        except Exception:
            continue
        raw.columns = [str(c).strip() for c in raw.columns]
        if "linea_qom" in raw.columns:   # skip Lineas tab
            continue
        qom_cols = [c for c in raw.columns if c.startswith("nombre_") and c.endswith("_qom")]
        for qc in qom_cols:
            ec = qc.replace("_qom", "_es")
            if ec not in raw.columns:
                continue
            sub = raw[[qc, ec]].dropna().copy()
            sub.columns = ["qom", "es"]
            sub["qom"] = sub["qom"].astype(str).str.strip()
            sub["es"]  = sub["es"].astype(str).str.strip()
            sub = sub[(sub["qom"] != "") & (sub["es"] != "")]
            sub = sub[(sub["qom"] != "nan") & (sub["es"] != "nan")]
            sub = sub[(sub["qom"].str.lower() != "n/a") & (sub["es"].str.lower() != "n/a")]
            if len(sub) > 0:
                sub["source_doc"] = doc_name
                pairs.append(sub[["qom", "es", "source_doc"]])
                print(f"  [META] {path.name} / {sheet} / {qc}: {len(sub)} pairs")
    return pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame()

xlsx_files = sorted(DATA_DIR.rglob("*.xlsx"))
print(f"Found {len(xlsx_files)} xlsx files:")
for p in xlsx_files:
    print(f"  {p.name}")

frames = []
for p in xlsx_files:
    print(f"\nLoading {p.name} ...")
    lines = read_workbook(p)
    meta  = read_metadata_names(p)
    if len(lines):
        frames.append(lines)
        print(f"  lines: {len(lines)}")
    if len(meta):
        frames.append(meta)
        print(f"  meta:  {len(meta)}")

all_df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
print(f"\nTotal from xlsx: {len(all_df)} rows")


Found 4 xlsx files:
  Arte verbal qom.xlsx
  Educacion Sanitaria Intercultural.xlsx
  Las Aventuras de Copaic.xlsx
  Materiales del Taller de Lengua y Cultura Toba.xlsx

Loading Arte verbal qom.xlsx ...
  [META] Arte verbal qom.xlsx / Capítulos / nombre_capitulo_qom: 4 pairs
  [META] Arte verbal qom.xlsx / Fragmentos / nombre_fragmento_qom: 62 pairs
  lines: 1825
  meta:  66

Loading Educacion Sanitaria Intercultural.xlsx ...
  [META] Educacion Sanitaria Intercultural.xlsx / Capítulos / nombre_capitulo_qom: 3 pairs
  [META] Educacion Sanitaria Intercultural.xlsx / Fragmentos / nombre_fragmento_qom: 42 pairs
  lines: 169
  meta:  45

Loading Las Aventuras de Copaic.xlsx ...
  lines: 16

Loading Materiales del Taller de Lengua y Cultura Toba.xlsx ...
  [META] Materiales del Taller de Lengua y Cultura Toba.xlsx / Secciones / nombre_seccion_qom: 3 pairs
  [META] Materiales del Taller de Lengua y Cultura Toba.xlsx / Fragmentos / nombre_fragmento_qom: 8 pairs
  lines: 267
  meta:  11

Total 

In [4]:
# Cell 4: Load CSV sources (El Principito + La Biblia)

def read_csv_source(path, qom_col, es_col, doc_name):
    raw = pd.read_csv(path)
    raw.columns = [str(c).strip() for c in raw.columns]
    sub = raw[[qom_col, es_col]].copy()
    sub.columns = ["qom", "es"]
    sub = sub.dropna(subset=["qom", "es"])
    sub["qom"] = sub["qom"].astype(str).str.strip()
    sub["es"]  = sub["es"].astype(str).str.strip()
    sub = sub[(sub["qom"] != "") & (sub["es"] != "")]
    sub = sub[(sub["qom"] != "nan") & (sub["es"] != "nan")]
    sub["source_doc"] = doc_name
    print(f"  {doc_name}: {len(sub)} pairs")
    return sub[["qom", "es", "source_doc"]]

csv_frames = []
pp_path = DATA_DIR / "El Principito.csv"
if pp_path.exists():
    csv_frames.append(read_csv_source(pp_path, "qom", "espanol", "El Principito"))
else:
    print(f"[WARN] Not found: {pp_path}")

biblia_path = DATA_DIR / "La Biblia.csv"
if biblia_path.exists():
    csv_frames.append(read_csv_source(biblia_path, "LNLE13", "DHHS94", "La Biblia"))
else:
    print(f"[WARN] Not found: {biblia_path}")

if csv_frames:
    all_df = pd.concat([all_df] + csv_frames, ignore_index=True)

print(f"\nTotal rows (xlsx + CSV): {len(all_df)}")


  El Principito: 557 pairs
  La Biblia: 30580 pairs

Total rows (xlsx + CSV): 33536


In [5]:
# Cell 5: QC — exact match with training notebook
#
#   lr > 0.15 and lr < 6.0  (exclusive)
#   char length < 800 per side
#   dedup on (qom, es, source_doc)  — cross-doc duplicates kept

_tok_qc = re.compile(r"\w+|[^\w\s]", re.UNICODE)

def tok_count(s):
    return len([t for t in _tok_qc.findall(s) if t.strip()])

all_df["qom_len_c"] = all_df["qom"].str.len()
all_df["es_len_c"]  = all_df["es"].str.len()
all_df["qom_len_t"] = all_df["qom"].apply(tok_count)
all_df["es_len_t"]  = all_df["es"].apply(tok_count)
all_df["lr"]        = (all_df["es_len_t"] + 1) / (all_df["qom_len_t"] + 1)

before = len(all_df)
all_df = all_df.drop_duplicates(subset=["qom", "es", "source_doc"], keep="first")
print(f"Dedup (qom, es, source_doc): {before} -> {len(all_df)}  (removed {before - len(all_df)})")

all_df = all_df[(all_df["lr"] > 0.15) & (all_df["lr"] < 6.0)]
all_df = all_df[(all_df["qom_len_c"] < 800) & (all_df["es_len_c"] < 800)]

print(f"After LR + char filters: {len(all_df)} rows")
print("\nPer document:")
print(all_df["source_doc"].value_counts().to_string())


Dedup (qom, es, source_doc): 33536 -> 33362  (removed 174)
After LR + char filters: 33331 rows

Per document:
source_doc
La Biblia                                         30449
Arte verbal qom                                    1831
El Principito                                       550
Materiales del Taller de Lengua y Cultura Toba      273
Educacion Sanitaria Intercultural                   212
Las Aventuras de Copaic                              16


In [6]:
# Cell 6: Compute and display statistics

rows = []
for doc, grp in all_df.groupby("source_doc"):
    qom_s = seg_stats(grp["qom"])
    es_s  = seg_stats(grp["es"])
    rows.append({
        "Corpus":      doc,
        "Segments":    len(grp),
        "QOM tokens":  qom_s["tokens"],
        "QOM avg len": qom_s["avg_len"],
        "QOM std len": qom_s["std_len"],
        "ES tokens":   es_s["tokens"],
        "ES avg len":  es_s["avg_len"],
        "ES std len":  es_s["std_len"],
    })

stats_df = pd.DataFrame(rows)

qom_all_lens = all_df["qom"].dropna().apply(lambda x: len(tokenize(x)))
es_all_lens  = all_df["es"].dropna().apply(lambda x: len(tokenize(x)))

total = {
    "Corpus":      "TOTAL",
    "Segments":    stats_df["Segments"].sum(),
    "QOM tokens":  int(qom_all_lens.sum()),
    "QOM avg len": round(float(qom_all_lens.mean()), 2),
    "QOM std len": round(float(qom_all_lens.std()),  2),
    "ES tokens":   int(es_all_lens.sum()),
    "ES avg len":  round(float(es_all_lens.mean()), 2),
    "ES std len":  round(float(es_all_lens.std()),  2),
}
stats_df = pd.concat([stats_df, pd.DataFrame([total])], ignore_index=True)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
print(stats_df.to_string(index=False))


                                        Corpus  Segments  QOM tokens  QOM avg len  QOM std len  ES tokens  ES avg len  ES std len
                               Arte verbal qom      1831       16242         8.87         5.44      15009        8.20        4.67
             Educacion Sanitaria Intercultural       212        7438        35.08        35.18       6343       29.92       30.58
                                 El Principito       550       16817        30.58        25.98      15793       28.71       26.44
                                     La Biblia     30449     1422512        46.72        21.14     849684       27.91       12.10
                       Las Aventuras de Copaic        16         560        35.00        26.95        484       30.25       20.95
Materiales del Taller de Lengua y Cultura Toba       273        2299         8.42         4.97       1946        7.13        3.42
                                         TOTAL     33331     1465868        43.98        2

In [7]:
# Cell 7: Export
out_path = OUT_DIR / "corpus_statistics.csv"
stats_df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


Saved: /kaggle/working/corpus_statistics.csv
